In [1]:
import pandas as pd

In [4]:
delay_df = pd.read_excel('data/Fleet Dealy Data from P2301 to P2611.xlsx')

start_date = '2025-12-01'
end_date = '2025-12-31'

# creating subset
baseline_df = delay_df[(delay_df['Date'] >= start_date) & (delay_df['Date'] <= end_date)]

print(f"Original rows: {len(delay_df)}")
print(f"rows for baseline model: {len(baseline_df)}")

Original rows: 3925
rows for baseline model: 87


In [9]:
weather_df = pd.read_csv('data/metro_weather_dec25.csv', skiprows=3, parse_dates=['time'])

weather_df = weather_df.rename(columns={'time':'Date'})

merge_df = pd.merge(baseline_df, weather_df, on='Date', how='inner')

In [14]:
# cleaning the df a little
cols_to_keep = [
    'Date',
    'Incid. Location',
    'Equip.',
    'temperature_2m (°C)',
    'precipitation (mm)',
    'wind_speed_10m (km/h)',
    'Delay Mins'
]

clean_df = merge_df[cols_to_keep].copy()

In [15]:
clean_df = clean_df.rename(columns={
    'Incid. Location': 'Location',
    'Equip.': 'Train_Equip',
    'temperature_2m (°C)': 'Temperature',
    'precipitation (mm)': 'Precipitation',
    'wind_speed_10m (km/h)': 'Wind_speed',
    'Delay Mins': 'Delay_Minutes'
})

In [16]:
clean_df = clean_df.dropna(subset=['Delay_Minutes'])

clean_df['Temperature'] = clean_df['Temperature'].fillna(0)
clean_df['Precipitation'] = clean_df['Precipitation'].fillna(0)
clean_df['Wind_speed'] = clean_df['Wind_speed'].fillna(0)

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [18]:
features = [
    'Precipitation',
    'Wind_speed',
    'Temperature',
]

X = clean_df[features]
y = clean_df['Delay_Minutes'].fillna(0)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [19]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)


print(f"MAE: {mae:.2f} minutes")

MAE: 20.38 minutes
